In [1]:
import os
import json
import pickle
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# =========================
# 0) utils
# =========================
def ensure_dir(p: str) -> str:
    os.makedirs(p, exist_ok=True)
    return p


def rename_pod2service(pod_name: str) -> str:
    return pod_name.replace("2-0", "").replace("-0", "").replace("-1", "").replace("-2", "")


def merge_entity(resource_type: str, entity: str) -> str:
    """
    Mirror HolisticRCA MetricGenerator.merge_entity
    """
    if resource_type in ("container", "istio"):
        return entity.replace("2-0", "").replace("-0", "").replace("-1", "").replace("-2", "")
    if resource_type == "node":
        return entity.split("-")[0]
    return entity


def calculate_statistic(arr_like) -> Dict[str, float]:
    """
    Mirror HolisticRCA MetricGenerator.calculate_statistic
    """
    metric_data = np.array(arr_like, dtype=float)
    median = float(np.nanmedian(metric_data)) if metric_data.size else np.nan
    p1 = float(np.nanpercentile(metric_data, 1)) if metric_data.size else np.nan
    p99 = float(np.nanpercentile(metric_data, 99)) if metric_data.size else np.nan
    q1 = float(np.nanpercentile(metric_data, 25)) if metric_data.size else np.nan
    q3 = float(np.nanpercentile(metric_data, 75)) if metric_data.size else np.nan
    mean = float(np.nanmean(metric_data)) if metric_data.size else np.nan
    std = float(np.nanstd(metric_data)) if metric_data.size else np.nan

    if metric_data.size and not (np.isnan(p1) or np.isnan(p99)):
        clip_data = np.clip(metric_data, p1, p99)
        clip_mean = float(np.nanmean(clip_data))
        clip_std = float(np.nanstd(clip_data))
    else:
        clip_mean, clip_std = np.nan, np.nan

    valid_ratio = float(np.count_nonzero(~np.isnan(metric_data)) / len(metric_data)) if metric_data.size else 0.0

    return {
        "clip_mean": clip_mean,
        "clip_std": clip_std,
        "percentile_1": p1,
        "q1": q1,
        "median": median,
        "q3": q3,
        "percentile_99": p99,
        "valid_ratio": valid_ratio,
        "mean": mean,
        "std": std,
    }


def diff_metric(metric_name: str, metric_data) -> np.ndarray:
    """
    Mirror HolisticRCA MetricGenerator.diff_metric
    (Your extracted metrics are already aligned per timestamp; diff is applied before stats/zscore.)
    """
    diff_name_list = [
        "Memory/system.mem.used",
        "Memory/system.mem.real.used",
        "Memory/system.mem.usable",
        "Disk/system.disk.free",
        "Disk/system.disk.used",
        "Memory/kpi_container_memory_usage_MB",
        "Memory/kpi_container_memory_working_set_MB",
        "Memory/kpi_container_memory_rss",
        "Memory/kpi_container_memory_mapped_file",
        "Disk/kpi_container_fs_reads_MB",
        "Disk/kpi_container_fs_usage_MB",
        "Disk/kpi_container_fs_writes_MB",
        "Thread/kpi_container_threads",
    ]
    arr = np.array(metric_data, dtype=float)
    if metric_name in diff_name_list and arr.size >= 2:
        # 差分处理，后一个元素减前一个元素，会导致结果少一个元素
        d = np.diff(arr)
        # 补齐少的元素
        d = np.append(d, d[-1])
        return d
    return arr


# =========================
# 1) list cases (same style as LogGenerator)
# =========================
def list_cases(raw_data_root: str) -> List[Tuple[str, str, str, str]]:
    """
    return [(dataset_type, date, cloudbed, raw_metric_dir), ...]
    raw_metric_dir = .../{dataset_type}/{date}/{cloudbed}/raw_metric
    """
    cases = []
    if not os.path.isdir(raw_data_root):
        raise FileNotFoundError(raw_data_root)

    for dataset_type in sorted(os.listdir(raw_data_root)):
        dt_path = os.path.join(raw_data_root, dataset_type)
        if not os.path.isdir(dt_path):
            continue
        for date in sorted(os.listdir(dt_path)):
            date_path = os.path.join(dt_path, date)
            if not os.path.isdir(date_path):
                continue
            for cloudbed in sorted(os.listdir(date_path)):
                cb_path = os.path.join(date_path, cloudbed)
                raw_metric_dir = os.path.join(cb_path, "raw_metric")
                if os.path.isdir(raw_metric_dir):
                    cases.append((dataset_type, date, cloudbed, raw_metric_dir))
    return cases


def load_metric_df(raw_metric_dir: str, resource_type: str, entity: str) -> pd.DataFrame:
    """
    raw_metric_dir/{resource_type}/{entity}.csv
    """
    fp = os.path.join(raw_metric_dir, resource_type, f"{entity}.csv")
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    return pd.read_csv(fp)


# =========================
# 2) MetricGenerator (script-style, mirrors original behavior)
# =========================
class AIOpsMetricGenerator:
    def __init__(
        self,
        temp_data_storage: str,
        raw_data_root: str,
        analysis_metric_dir: str,
        dataset_metric_dir: str,
        node_list: List[str],
        service_list: List[str],
        pod_list: List[str],
        window_size_list: List[int],
        time_interval_dir: str,   # ✅ 新增：time_interval pkl 所在目录
        skip_bad_case: bool = True,
        bad_case_date: str = "2022-03-24",
        bad_case_cloudbeds: Tuple[str, ...] = ("cloudbed-1", "cloudbed-2"),
        # optional: for fallback windows, pass a loader for log timestamp axis
        log_timestamp_loader=None,
    ):
        
        self.time_interval_dir = time_interval_dir  # ✅ 保存
        self.temp_data_storage = temp_data_storage
        self.raw_data_root = raw_data_root
        self.analysis_metric_dir = ensure_dir(analysis_metric_dir)
        self.dataset_metric_dir = ensure_dir(dataset_metric_dir)

        self.node_list = node_list
        self.service_list = service_list
        self.pod_list = pod_list

        self.window_size_list = window_size_list

        self.skip_bad_case = skip_bad_case
        self.bad_case_date = bad_case_date
        self.bad_case_cloudbeds = bad_case_cloudbeds

        self.log_timestamp_loader = log_timestamp_loader  # function(raw_log_dir)->DataFrame, used for fallback

        # try TimeIntervalLabelGenerator
        self._has_time_interval = False
        self._TimeIntervalLabelGenerator = None
        try:
            from data_filter.CCF_AIOps_challenge_2022.service.time_interval_label_generator import (
                TimeIntervalLabelGenerator,
            )

            self._TimeIntervalLabelGenerator = TimeIntervalLabelGenerator
            self._has_time_interval = True
        except Exception:
            self._has_time_interval = False

    # -------------------------
    # 2.1 build common statistics (train only)
    # -------------------------
    def calculate_common_statistic(self):
        cases = list_cases(self.raw_data_root)

        common_stat: Dict[str, Dict[str, Dict[str, Any]]] = {}
        data_dict: Dict[str, Dict[str, Dict[str, List[float]]]] = {}

        resource_types = ["node", "service", "container", "istio"]

        for dataset_type, date, cloudbed, raw_metric_dir in tqdm(cases, desc="scan metric cases"):
            if dataset_type == "test":
                continue
            if self.skip_bad_case and date == self.bad_case_date and cloudbed in self.bad_case_cloudbeds:
                continue

            for rt in resource_types:
                common_stat.setdefault(rt, {})
                data_dict.setdefault(rt, {})

                if rt == "node":
                    entities = self.node_list
                elif rt == "service":
                    entities = self.service_list
                else:
                    entities = self.pod_list

                for ent in entities:
                    # node解析为node;pod解析为微服务名
                    merged_ent = merge_entity(rt, ent)
                    common_stat[rt].setdefault(merged_ent, {})
                    data_dict[rt].setdefault(merged_ent, {})

                    df = load_metric_df(raw_metric_dir, rt, ent)
                    for col in df.columns:
                        if col == "timestamp":
                            continue
                        # common_stat["container"]["adservice"]["CPU/kpi_container_cpu_usage_seconds"] 统计值
                        common_stat[rt][merged_ent].setdefault(col, 0)
                        # data_dict["container"]["adservice"]["CPU/kpi_container_cpu_usage_seconds"] 数据
                        data_dict[rt][merged_ent].setdefault(col, [])

                        series = df[col].to_numpy(dtype=float)
                        # 对部分指标，处理数据为差分序列
                        series = diff_metric(col, series)
                        data_dict[rt][merged_ent][col].extend(series.tolist())

        # compute statistic
        # 计算合并实体，不同metric指标的统计值
        for rt in common_stat.keys():
            for merged_ent in common_stat[rt].keys():
                for metric_name in list(common_stat[rt][merged_ent].keys()):
                    common_stat[rt][merged_ent][metric_name] = calculate_statistic(
                        data_dict[rt][merged_ent][metric_name]
                    )

        out_path = os.path.join(self.analysis_metric_dir, "common_statistic.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(common_stat, f, indent=2)
        print("[OK] saved:", out_path)
        return common_stat

    # -------------------------
    # 2.2 z-score all cases -> nested dict of DataFrames
    # -------------------------
    def z_score_metric_data(self):
        stat_path = os.path.join(self.analysis_metric_dir, "common_statistic.json")
        with open(stat_path, "r", encoding="utf-8") as f:
            common_stat = json.load(f)

        cases = list_cases(self.raw_data_root)
        out: Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, pd.DataFrame]]]]] = {}
        # out[dataset_type][date][cloudbed][resource_type][entity] = DataFrame

        resource_types = ["node", "service", "container", "istio"]

        for dataset_type, date, cloudbed, raw_metric_dir in tqdm(cases, desc="z-score metric"):
            if self.skip_bad_case and date == self.bad_case_date and cloudbed in self.bad_case_cloudbeds:
                continue

            out.setdefault(dataset_type, {}).setdefault(date, {}).setdefault(cloudbed, {})

            for rt in resource_types:
                out[dataset_type][date][cloudbed].setdefault(rt, {})

                if rt == "node":
                    entities = self.node_list
                elif rt == "service":
                    entities = self.service_list
                else:
                    entities = self.pod_list

                for ent in entities:
                    merged_ent = merge_entity(rt, ent)
                    df = load_metric_df(raw_metric_dir, rt, ent).copy()

                    for col in df.columns:
                        if col == "timestamp":
                            continue
                        raw = df[col].to_numpy(dtype=float)
                        raw = diff_metric(col, raw)

                        mean = common_stat.get(rt, {}).get(merged_ent, {}).get(col, {}).get("mean", np.nan)
                        std = common_stat.get(rt, {}).get(merged_ent, {}).get(col, {}).get("std", np.nan)

                        if not (np.isnan(mean) or np.isnan(std) or std == 0):
                            z = (raw - mean) / std
                            z = np.nan_to_num(z, nan=0.0)
                        else:
                            z = np.zeros_like(raw, dtype=float)

                        df[col] = z

                    out[dataset_type][date][cloudbed][rt][ent] = df

        out_path = os.path.join(self.dataset_metric_dir, "all_metric.pkl")
        with open(out_path, "wb") as f:
            pickle.dump(out, f)
        print("[OK] saved:", out_path)
        return out

    # -------------------------
    # 2.3 time intervals
    # -------------------------
    def _fallback_time_intervals(
        self,
        window_size_min: int,
        dataset_type: str,
        cases: List[Tuple[str, str, str, str]],
    ) -> List[Tuple[str, str, int, int]]:
        """
        fallback like your LogGenerator:
        use timestamp axis from container log features (requires log_timestamp_loader)
        interval tuple: (date, cloudbed, st, et)
        """
        if self.log_timestamp_loader is None:
            raise RuntimeError("Fallback needs log_timestamp_loader(raw_log_dir)->DataFrame with 'timestamp'.")

        case_map: Dict[str, Dict[str, Dict[str, str]]] = {}
        for dt, date, cb, raw_metric_dir in cases:
            case_map.setdefault(dt, {}).setdefault(date, {})[cb] = raw_metric_dir

        out = []
        for date, cb_dict in case_map.get(dataset_type, {}).items():
            for cloudbed, raw_metric_dir in cb_dict.items():
                # raw_metric_dir sibling raw_log dir
                raw_log_dir = os.path.join(os.path.dirname(raw_metric_dir), "raw_log")
                df_log = self.log_timestamp_loader(raw_log_dir)
                ts = df_log["timestamp"].values
                for i in range(0, len(ts) - window_size_min):
                    st = int(ts[i])
                    et = int(ts[i + window_size_min])
                    out.append((date, cloudbed, st, et))
        return out

    # 全天的老版本
    # def _get_time_interval_list(
    #     self,
    #     window_size: int,
    #     data_type: str,
    #     cases: List[Tuple[str, str, str, str]],
    # ):
    #     if self._has_time_interval and self._TimeIntervalLabelGenerator is not None:
    #         gen = self._TimeIntervalLabelGenerator()
    #         return gen.get_time_interval_label(window_size)["time_interval"][data_type]

    #     dataset_type = "training_data_with_faults" if data_type == "train_valid" else "test_data"
    #     return self._fallback_time_intervals(window_size, dataset_type, cases)
    
    def _get_time_interval_list(
        self,
        window_size: int,
        data_type: str,  # "train_valid" or "test"
        cases: List[Tuple[str, str, str, str]],
    ):
        """
        ✅ 与 log/trace 对齐：直接使用 label generator 产出的 time_interval_window_size_{w}.pkl
        返回: List[(date, cloudbed, st, et)]
        """
        pkl_path = os.path.join(self.time_interval_dir, f"time_interval_window_size_{window_size}.pkl")
        if not os.path.exists(pkl_path):
            raise FileNotFoundError(
                f"[metric] time interval pkl not found: {pkl_path}\n"
                f"Please run label generator first."
            )

        with open(pkl_path, "rb") as f:
            obj = pickle.load(f)

        intervals = obj["time_interval"][data_type]  # list like [date, cloudbed, st, et]
        return [(x[0], x[1], int(x[2]), int(x[3])) for x in intervals]    

    # -------------------------
    # 2.4 generate windowed metric dataset
    # -------------------------
    @staticmethod
    def _slice(st: int, et: int, df: pd.DataFrame) -> np.ndarray:
        return np.array(df.query(f"{st} <= timestamp < {et}").iloc[:, df.columns != "timestamp"].values)

    def generate_metric_data(self):
        all_metric_path = os.path.join(self.dataset_metric_dir, "all_metric.pkl")
        with open(all_metric_path, "rb") as f:
            all_metric = pickle.load(f)

        # flatten as original MetricGenerator does: all_metric_dict[date][cloudbed]
        all_metric_dict: Dict[str, Dict[str, Any]] = {}
        for date_cloudbed_data in all_metric.values():
            for date, cloudbed_data in date_cloudbed_data.items():
                all_metric_dict[date] = cloudbed_data

        cases = list_cases(self.raw_data_root)

        for window_size in tqdm(self.window_size_list, desc="metric window_size"):
            metric_dict = {}
            entity_features: List[Tuple[str, Tuple[int, int]]] = []
            metric_name_list: List[str] = []
            record_features = True

            for data_type in ["train_valid", "test"]:
                intervals = self._get_time_interval_list(window_size, data_type, cases)
                metric_dict[data_type] = []

                for time_interval in tqdm(intervals, desc=f"{data_type} intervals", leave=False):
                    date, cloudbed, st, et = time_interval[0], time_interval[1], int(time_interval[2]), int(time_interval[3])

                    feature_index = 0
                    metric_data = all_metric_dict[date][cloudbed]  # dict with rt->entity->df

                    # ---- node ----
                    node_data_list = []
                    for node in self.node_list:
                        df = metric_data["node"][node]
                        # (T, node_feature_num)
                        temp = self._slice(st, et, df)
                        '''
                            node_data_list = [
                            node1_matrix,
                            node2_matrix,
                            node3_matrix,
                            ...
                            ]
                        '''
                        node_data_list.append(temp)

                        if record_features:
                            '''
                                node-1 : (0, 24)
                                node-2 : (24, 48)
                                node-3 : (48, 72)
                            '''
                            entity_features.append((node, (feature_index, feature_index + temp.shape[-1])))
                            feature_index += temp.shape[-1]
                            cols = list(df.columns[df.columns != "timestamp"])
                            '''
                                特征名列表:
                                node-1/CPU/system.load.1
                                node-1/CPU/system.load.5
                                node-1/Memory/system.mem.used
                            '''
                            metric_name_list.extend([f"{node}/{c}" for c in cols])
                    # (T, all_node_feature_num)
                    node_data = np.concatenate(node_data_list, axis=-1) if node_data_list else np.empty((0, 0))

                    # ---- service ----
                    svc_data_list = []
                    for svc in self.service_list:
                        df = metric_data["service"][svc]
                        temp = self._slice(st, et, df)
                        svc_data_list.append(temp)

                        if record_features:
                            entity_features.append((svc, (feature_index, feature_index + temp.shape[-1])))
                            feature_index += temp.shape[-1]
                            cols = list(df.columns[df.columns != "timestamp"])
                            metric_name_list.extend([f"{svc}/{c}" for c in cols])
                    svc_data = np.concatenate(svc_data_list, axis=-1) if svc_data_list else np.empty((node_data.shape[0], 0))

                    # ---- pod/container ----
                    pod_data_list = []
                    for pod in self.pod_list:
                        df = metric_data["container"][pod]
                        temp = self._slice(st, et, df)
                        pod_data_list.append(temp)

                        if record_features:
                            entity_features.append((pod, (feature_index, feature_index + temp.shape[-1])))
                            feature_index += temp.shape[-1]
                            cols = list(df.columns[df.columns != "timestamp"])
                            metric_name_list.extend([f"{pod}/{c}" for c in cols])
                    pod_data = np.concatenate(pod_data_list, axis=-1) if pod_data_list else np.empty((node_data.shape[0], 0))

                    metric_item = np.concatenate((node_data, svc_data, pod_data), axis=-1)
                    metric_dict[data_type].append(metric_item)
                    record_features = False

            out_path = os.path.join(self.dataset_metric_dir, f"metric_window_size_{window_size}.pkl")
            '''
                metric_data = {
                    "train_valid": [
                        window_matrix_1,
                        window_matrix_2,
                        ...
                    ],
                    "test": [
                        window_matrix_1,
                        window_matrix_2,
                        ...
                    ]
                }
                每一个 window_matrix形状为(T_window, feature_dim)
                ================================================
                entity_features = [
                    (entity_name, (start_index, end_index)),
                    ...
                ]

                [
                    ('node-1', (0,24)),
                    ('node-2', (24,48)),
                    ...
                    ('adservice', (144,160)),
                    ...
                    ('adservice-0', (320,340)),
                    ...
                ]
                =================================================
                metric_names为指标名
                [
                    "node-1/CPU/system.load.1",
                    "node-1/CPU/system.load.5",
                    ...
                    "node-2/CPU/system.load.1",
                    ...
                    "adservice/Request/rr.grpc",
                    ...
                    "adservice-0/CPU/kpi_container_cpu_usage_seconds",
                    ...
                ]
            '''
            with open(out_path, "wb") as f:
                pickle.dump(
                    {
                        "metric_data": metric_dict,
                        "entity_features": entity_features,
                        "metric_names": metric_name_list,
                    },
                    f,
                )
            print("[OK] saved:", out_path)

    # helper getter like original
    def get_metric(self, window_size: int) -> dict:
        fp = os.path.join(self.dataset_metric_dir, f"metric_window_size_{window_size}.pkl")
        with open(fp, "rb") as f:
            return pickle.load(f)


# =========================
# 3) Example main (fill your paths & lists)
# =========================
if __name__ == "__main__":
    # ---- paths (match your LogGenerator variables) ----
    TEMP_DATA_STORAGE = "../../temp_data/aiops22"
    RAW_DATA_ROOT = f"{TEMP_DATA_STORAGE}/raw_data"
    ANALYSIS_METRIC_DIR = f"{TEMP_DATA_STORAGE}/analysis/metric"
    DATASET_METRIC_DIR = f"{TEMP_DATA_STORAGE}/dataset/metric"
    TIME_INTERVAL_DIR = f"{TEMP_DATA_STORAGE}/dataset/time_interval_and_label"

    # ---- entity lists ----
    pod_list = [
        'adservice-0', 
        'adservice-1', 
        'adservice-2', 
        'adservice2-0', 
        'cartservice-0', 
        'cartservice-1', 
        'cartservice-2', 
        'cartservice2-0', 
        'checkoutservice-0', 
        'checkoutservice-1', 
        'checkoutservice-2', 
        'checkoutservice2-0', 
        'currencyservice-0', 
        'currencyservice-1', 
        'currencyservice-2', 
        'currencyservice2-0', 
        'emailservice-0', 
        'emailservice-1', 
        'emailservice-2', 
        'emailservice2-0', 
        'frontend-0', 
        'frontend-1', 
        'frontend-2', 
        'frontend2-0', 
        'paymentservice-0', 
        'paymentservice-1', 
        'paymentservice-2', 
        'paymentservice2-0', 
        'productcatalogservice-0', 
        'productcatalogservice-1', 
        'productcatalogservice-2', 
        'productcatalogservice2-0', 
        'recommendationservice-0', 
        'recommendationservice-1', 
        'recommendationservice-2', 
        'recommendationservice2-0', 
        'shippingservice-0', 
        'shippingservice-1', 
        'shippingservice-2', 
        'shippingservice2-0'
    ]
    service_order_list = [
        "adservice","cartservice","checkoutservice","currencyservice","emailservice",
        "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
    ]
    node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

    # window sizes (minutes, aligned with TimeIntervalLabelGenerator)
    window_size_list = [5, 10, 15]

    # ---- optional fallback uses log timestamps ----
    def load_container_log_features(raw_log_dir: str) -> pd.DataFrame:
        f = os.path.join(raw_log_dir, "log_patterns_count.csv")
        return pd.read_csv(f)

    gen = AIOpsMetricGenerator(
        temp_data_storage=TEMP_DATA_STORAGE,
        raw_data_root=RAW_DATA_ROOT,
        analysis_metric_dir=ANALYSIS_METRIC_DIR,
        dataset_metric_dir=DATASET_METRIC_DIR,
        node_list=node_list,
        service_list=service_order_list,
        pod_list=pod_list,
        window_size_list=window_size_list,
        time_interval_dir=TIME_INTERVAL_DIR,      # ✅ 新增这一行
        log_timestamp_loader=load_container_log_features,  # needed only if TimeIntervalLabelGenerator unavailable
    )

    gen.calculate_common_statistic()
    # 数据变换标准化
    gen.z_score_metric_data()
    gen.generate_metric_data()

    # quick check
    _ = gen.get_metric(5)
    print("[OK] loaded metric window=5:", _.keys())

scan metric cases: 100%|██████████| 15/15 [00:04<00:00,  3.04it/s]
/tmp/ipykernel_3413591/1252917162.py:39: RuntimeWarning: All-NaN slice encountered
  median = float(np.nanmedian(metric_data)) if metric_data.size else np.nan
/home/lenovo/miniconda3/envs/HolisticRCA/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/tmp/ipykernel_3413591/1252917162.py:44: RuntimeWarning: Mean of empty slice
  mean = float(np.nanmean(metric_data)) if metric_data.size else np.nan
/home/lenovo/miniconda3/envs/HolisticRCA/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
z-score metric:   0%|          | 0/15 [00:00<?, ?it/s]

[OK] saved: ../../temp_data/aiops22/analysis/metric/common_statistic.json


z-score metric: 100%|██████████| 15/15 [00:04<00:00,  3.13it/s]


[OK] saved: ../../temp_data/aiops22/dataset/metric/all_metric.pkl


train_valid intervals: 100%|██████████| 300/300 [00:42<00:00,  7.14it/s]
                                                                        
train_valid intervals:   0%|          | 1/300 [00:00<00:46,  6.46it/s]

[OK] saved: ../../temp_data/aiops22/dataset/metric/metric_window_size_5.pkl



train_valid intervals: 100%|██████████| 300/300 [00:42<00:00,  7.11it/s]
                                                                        
train_valid intervals:   0%|          | 1/300 [00:00<00:42,  7.08it/s]

[OK] saved: ../../temp_data/aiops22/dataset/metric/metric_window_size_10.pkl



train_valid intervals: 100%|██████████| 300/300 [00:42<00:00,  7.18it/s]
                                                                        
metric window_size: 100%|██████████| 3/3 [03:48<00:00, 76.23s/it]

[OK] saved: ../../temp_data/aiops22/dataset/metric/metric_window_size_15.pkl
[OK] loaded metric window=5: dict_keys(['metric_data', 'entity_features', 'metric_names'])
